# Controlled graph-event streams for QRC

This notebook runs the same package API as the CLI. It covers dataset construction, FINAL-compatible QRC/CRC/ESN features, paired linear readout, statistics, and visualization. The default is a small CPU smoke test; change `PROFILE` to `quick` or `full` for experiments.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, display

MODULE_ROOT = Path.cwd()
if not (MODULE_ROOT / 'controlled_streams').exists():
    MODULE_ROOT = Path('/home/ubuntu/Workspace/Fei/wlf/Article/QRC/FINAL/6_controlled_graph_streams')
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

from controlled_streams import RunConfig, available_devices, generate_dataset, run_experiment, task_settings

## Configuration

All editable controls are here. The canonical scientific configuration is `W=0.6`, `g=0.84`, and `L=3`.

In [ ]:
PROFILE = 'smoke'  # 'smoke', 'quick', or 'full'
RUN_EXPERIMENT = True
TASK_FILTER = None  # e.g. {'direction', 'delayed_motif'}; None runs the profile manifest
OUTPUT_ROOT = MODULE_ROOT / 'outputs'
DEVICES = available_devices() if PROFILE != 'smoke' else ('cpu',)

cfg = RunConfig.for_profile(
    PROFILE,
    output_root=str(OUTPUT_ROOT),
    devices=DEVICES,
    train_device=DEVICES[0],
)
cfg

## Manifest and input-control audit

In [ ]:
settings = task_settings(PROFILE)
if TASK_FILTER:
    settings = [s for s in settings if s.family in TASK_FILTER or s.name in TASK_FILTER]
manifest = pd.DataFrame([s.to_dict() for s in settings])
display(manifest)

example = generate_dataset(settings[0], cfg.q, cfg.n_groups, cfg.generation_seed)
print('Example tensor:', example.tokens.shape)
print('Class counts:', example.metadata['class_counts'])
print('Control:', example.metadata['controlled_claim'])

## Train and evaluate

Settings run sequentially. Reservoir seeds are striped across `DEVICES` within each setting. Existing compatible outputs are resumed unless `force=True` is set in the configuration.

In [ ]:
if RUN_EXPERIMENT:
    metrics, summary, paired = run_experiment(cfg, settings)
else:
    profile_root = OUTPUT_ROOT / PROFILE
    metrics = pd.read_csv(profile_root / 'metrics_all_settings.csv')
    summary = pd.read_csv(profile_root / 'summary_all_settings.csv')
    paired = pd.read_csv(profile_root / 'paired_statistics_all_settings.csv')
print('Metric rows:', len(metrics), 'Paired comparisons:', len(paired))

## Performance summary

In [ ]:
display(
    summary[['setting', 'family', 'method', 'test_auc_mean', 'test_auc_std', 'test_acc_mean']]
    .sort_values(['family', 'setting', 'test_auc_mean'], ascending=[True, True, False])
)

## Seed-paired QRC comparisons

The inferential unit is the paired seed, not the number of synthetic samples. Smoke and quick profiles are diagnostics; only the ten-seed full profile is intended for inference.

In [ ]:
display(
    paired[paired['metric'] == 'test_auc'][[
        'setting', 'comparison', 'n_pairs', 'mean_difference',
        'bootstrap_ci_low', 'bootstrap_ci_high', 'wins', 'ties', 'losses',
        'paired_t_p', 'wilcoxon_p'
    ]].sort_values(['setting', 'comparison'])
)

## Saved visualizations

In [ ]:
figure_dir = OUTPUT_ROOT / PROFILE / 'figures'
for path in sorted(figure_dir.glob('*.png')):
    print(path.name)
    display(Image(filename=str(path)))